# NBA Breakout Analysis
**Author:** Mike Zhang  
**Date:** 2026-03-19  
**Purpose:** Identify players who broke out in the second half of seasons on non-playoff teams and examine whether their post all-star performance from the previous season predicts their following season performance

In [18]:
import pandas as pd
from nba_api.stats.endpoints import leaguedashplayerstats
from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.endpoints import commonplayerinfo    
from nba_api.stats.endpoints import drafthistory
from pathlib import Path


In [2]:
def get_breakout_candidates(season):
    # Construct season strings
    start_year = int(season.split('-')[0])
    following_season_str = f"{start_year + 1}-{str(start_year + 2)[-2:]}"
    previous_season_str = f"{start_year - 1}-{str(start_year)[-2:]}"
    
    # Pull post all-star player stats
    post = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_segment_nullable='Post All-Star',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull pre all-star player stats
    pre = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_segment_nullable='Pre All-Star',
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull following full season stats
    following = leaguedashplayerstats.LeagueDashPlayerStats(
        season=following_season_str,
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull previous full season stats
    previous = leaguedashplayerstats.LeagueDashPlayerStats(
        season=previous_season_str,
        measure_type_detailed_defense='Advanced'
    ).get_data_frames()[0]
    
    # Pull pre all-star team standings
    pre_allstar_teams = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_segment_nullable='Pre All-Star'
    ).get_data_frames()[0]
    
    # Identify tanking teams (bottom 12 by pre all-star win rate)
    tanking_teams = pre_allstar_teams.nsmallest(12, 'W_PCT')[['TEAM_ID', 'TEAM_NAME']]
    
    # Keep relevant columns
    split_cols = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 
                  'AGE', 'GP', 'MIN', 'PIE', 'USG_PCT']
    post = post[split_cols].copy()
    pre = pre[split_cols].copy()
    following = following[['PLAYER_ID', 'PIE', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP']].copy()
    previous = previous[['PLAYER_ID', 'PIE']].copy()
    
    # Apply post all-star filters
    post_filtered = post[
        (post['GP'] >= 15) &
        (post['MIN'] >= 25) &
        (post['AGE'] <= 28)
    ].copy()
    
    # Apply pre all-star filters
    pre_filtered = pre[
        (pre['GP'] >= 20) &
        (pre['MIN'] >= 10)
    ].copy()
    
    # Merge pre and post on PLAYER_ID
    merged = post_filtered.merge(
        pre_filtered,
        on='PLAYER_ID',
        suffixes=('_post', '_pre')
    )
    
    # Apply breakout criterion
    merged = merged[
        (merged['PIE_post'] - merged['PIE_pre'] >= 0.02) |
        (merged['MIN_post'] >= 1.3 * merged['MIN_pre'])
    ].copy()
    
    # Filter for tanking teams
    merged = merged[
        merged['TEAM_ID_post'].isin(tanking_teams['TEAM_ID'])
    ].copy()
    
    # Merge previous season PIE and filter out established players
    merged = merged.merge(
        previous[['PLAYER_ID', 'PIE']].rename(columns={'PIE': 'PIE_previous'}),
        on='PLAYER_ID',
        how='left'
    )
    
    # Keep rookies (no previous season data) or players with PIE <= 0.110
    merged = merged[
        (merged['PIE_previous'].isna()) |
        (merged['PIE_previous'] <= 0.110)
    ].copy()
    
    # Merge following season PIE and GP
    merged = merged.merge(
        following[['PLAYER_ID', 'PIE', 'GP', 'TEAM_ID' , 'TEAM_ABBREVIATION']].rename(columns={
            'PIE': 'PIE_following',
            'GP': 'GP_following',
            'TEAM_ABBREVIATION': 'TEAM_ABBREVIATION_following',
            'TEAM_ID': 'TEAM_ID_following'
        }),
        on='PLAYER_ID',
        how='left'
    )
    
    # Filter for minimum games played in following season
    merged = merged[
        merged['GP_following'] >= 30
    ].copy()
    
    # Add season label
    merged['season'] = season
    
    return merged

print("Function defined successfully")

Function defined successfully


In [3]:
test = get_breakout_candidates('2022-23')
print(test.shape)
print(test[['PLAYER_NAME_post','MIN_pre', 'MIN_post', 'PIE_post', 'PIE_pre', 'PIE_previous', 'PIE_following', 'TEAM_ID_post', 'TEAM_ID_following']].to_string())

(14, 23)
       PLAYER_NAME_post  MIN_pre  MIN_post  PIE_post  PIE_pre  PIE_previous  PIE_following  TEAM_ID_post  TEAM_ID_following
0       Andrew Nembhard     27.1      29.0     0.085    0.060           NaN          0.075    1610612754         1610612754
1         Austin Reaves     27.9      30.4     0.138    0.077         0.076          0.105    1610612747         1610612747
2            Coby White     22.3      25.9     0.110    0.076         0.080          0.099    1610612741         1610612741
3      Dennis Smith Jr.     25.0      27.4     0.105    0.084         0.095          0.092    1610612766         1610612751
6         James Wiseman     13.0      25.3     0.103    0.107           NaN          0.097    1610612765         1610612765
7          Jordan Nwora     15.6      25.7     0.099    0.078         0.072          0.103    1610612754         1610612761
8      Matisse Thybulle     12.7      27.6     0.057    0.052         0.057          0.061    1610612757         1610612757

In [4]:
import time

seasons = [
    '2014-15', '2015-16', '2016-17', '2017-18', '2018-19',
    '2019-20', '2020-21', '2021-22', '2022-23', '2023-24'
]

all_seasons = []

for season in seasons:
    print(f"Processing {season}...")
    try:
        df = get_breakout_candidates(season)
        all_seasons.append(df)
        print(f"{season}: {len(df)} candidates found")
    except Exception as e:
        print(f"{season}: Error - {e}")
    time.sleep(3)  # pause between API calls to avoid rate limiting

# Combine all seasons
all_candidates = pd.concat(all_seasons, ignore_index=True)
all_candidates = all_candidates[all_candidates['season'] != '2019-20'].copy()
print(f"\nTotal candidates across all seasons: {len(all_candidates)}")

Processing 2014-15...
2014-15: 18 candidates found
Processing 2015-16...
2015-16: 5 candidates found
Processing 2016-17...
2016-17: 9 candidates found
Processing 2017-18...
2017-18: 7 candidates found
Processing 2018-19...
2018-19: 8 candidates found
Processing 2019-20...
2019-20: Error - HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Processing 2020-21...
2020-21: 12 candidates found
Processing 2021-22...
2021-22: 16 candidates found
Processing 2022-23...
2022-23: 14 candidates found
Processing 2023-24...
2023-24: 11 candidates found

Total candidates across all seasons: 100


In [5]:
# Check distribution across seasons
print(all_candidates.groupby('season').size())

# Check for any missing PIE_following values
print("\nMissing PIE_following:", all_candidates['PIE_following'].isna().sum())

# Quick summary stats
print("\nPIE_post summary:")
print(all_candidates['PIE_post'].describe())

print("\nPIE_following summary:")
print(all_candidates['PIE_following'].describe())


season
2014-15    18
2015-16     5
2016-17     9
2017-18     7
2018-19     8
2020-21    12
2021-22    16
2022-23    14
2023-24    11
dtype: int64

Missing PIE_following: 0

PIE_post summary:
count    100.00000
mean       0.09981
std        0.02640
min        0.05200
25%        0.08200
50%        0.09950
75%        0.11775
max        0.17400
Name: PIE_post, dtype: float64

PIE_following summary:
count    100.000000
mean       0.091500
std        0.023069
min        0.039000
25%        0.074000
50%        0.092000
75%        0.107000
max        0.160000
Name: PIE_following, dtype: float64


In [6]:
from nba_api.stats.endpoints import leaguestandingsv3

def get_team_wins(season):
    standings = leaguestandingsv3.LeagueStandingsV3(
        season=season
    ).get_data_frames()[0]
    
    return standings[['TeamID', 'TeamCity', 'TeamName', 'WINS']].copy()

# Test it
test_wins = get_team_wins('2023-24')
print(test_wins.head())

       TeamID       TeamCity TeamName  WINS
0  1610612738         Boston  Celtics    64
1  1610612760  Oklahoma City  Thunder    57
2  1610612743         Denver  Nuggets    57
3  1610612752       New York   Knicks    50
4  1610612749      Milwaukee    Bucks    49


In [7]:
def get_three_year_avg_wins(season):
    start_year = int(season.split('-')[0])
    
    # Construct 3 preceding season strings
    prior_seasons = [
        f"{start_year - 3}-{str(start_year - 2)[-2:]}",
        f"{start_year - 2}-{str(start_year - 1)[-2:]}",
        f"{start_year - 1}-{str(start_year)[-2:]}"
    ]
    
    dfs = []
    for s in prior_seasons:
        df = get_team_wins(s)
        df['prior_season'] = s
        dfs.append(df)
        time.sleep(1)
    
    # Combine and average wins by team
    combined = pd.concat(dfs, ignore_index=True)
    avg_wins = combined.groupby('TeamID')['WINS'].mean().reset_index()
    avg_wins.columns = ['TEAM_ID', 'avg_wins_3yr']
    
    return avg_wins

# Test on 2023-24
test_avg = get_three_year_avg_wins('2023-24')
print(test_avg.sort_values('avg_wins_3yr', ascending=False).head(10))

       TEAM_ID  avg_wins_3yr
19  1610612756     53.333333
12  1610612749     51.666667
18  1610612755     51.333333
6   1610612743     49.333333
26  1610612763     48.333333
1   1610612738     48.000000
25  1610612762     46.000000
11  1610612748     45.666667
14  1610612751     45.666667
7   1610612744     45.333333


In [8]:
seasons_no_covid = [s for s in seasons if s != '2019-20']

avg_wins_all = []
for season in seasons_no_covid:
    print(f"Getting wins for {season}...")
    avg_wins = get_three_year_avg_wins(season)
    avg_wins['season'] = season
    avg_wins = avg_wins.rename(columns={'TEAM_ID': 'TEAM_ID_post'})
    avg_wins_all.append(avg_wins)
    time.sleep(1)

avg_wins_combined = pd.concat(avg_wins_all, ignore_index=True)

# Merge into all_candidates
all_candidates = all_candidates.merge(
    avg_wins_combined,
    on=['TEAM_ID_post', 'season'],
    how='left'
)

print("Missing avg_wins_3yr:", all_candidates['avg_wins_3yr'].isna().sum())
print(all_candidates[['PLAYER_NAME_post', 'season', 'avg_wins_3yr']].head(10).to_string())

Getting wins for 2014-15...
Getting wins for 2015-16...
Getting wins for 2016-17...
Getting wins for 2017-18...
Getting wins for 2018-19...
Getting wins for 2020-21...
Getting wins for 2021-22...
Getting wins for 2022-23...
Getting wins for 2023-24...
Missing avg_wins_3yr: 0
   PLAYER_NAME_post   season  avg_wins_3yr
0  Bojan Bogdanovic  2014-15     38.333333
1          CJ Miles  2014-15     49.000000
2    Chase Budinger  2014-15     32.333333
3  Danilo Gallinari  2014-15     43.666667
4     Elfrid Payton  2014-15     26.666667
5     Isaiah Canaan  2014-15     29.333333
6       Jae Crowder  2014-15     35.000000
7        Jeremy Lin  2014-15     37.666667
8   Jordan Clarkson  2014-15     37.666667
9      Nerlens Noel  2014-15     29.333333


In [9]:
all_candidates['PIE_change'] = all_candidates['PIE_post'] - all_candidates['PIE_pre']
all_candidates['USG_change'] = all_candidates['USG_PCT_post'] - all_candidates['USG_PCT_pre']
all_candidates['PIE_diff'] = all_candidates['PIE_following'] - all_candidates['PIE_post']

print(all_candidates[['PLAYER_NAME_post', 'PIE_change', 'USG_change', 'PIE_diff']].head(10).to_string())

   PLAYER_NAME_post  PIE_change  USG_change  PIE_diff
0  Bojan Bogdanovic       0.032       0.018    -0.020
1          CJ Miles       0.027      -0.012    -0.025
2    Chase Budinger       0.037       0.006    -0.031
3  Danilo Gallinari       0.026       0.030     0.000
4     Elfrid Payton       0.029       0.000    -0.017
5     Isaiah Canaan       0.003       0.036    -0.015
6       Jae Crowder       0.022       0.040     0.000
7        Jeremy Lin       0.037       0.038    -0.032
8   Jordan Clarkson       0.033       0.024    -0.034
9      Nerlens Noel       0.051       0.035    -0.027


In [10]:
positions = []

for player_id in all_candidates['PLAYER_ID'].unique():
    try:
        info = commonplayerinfo.CommonPlayerInfo(
            player_id=player_id
        ).get_data_frames()[0]
        positions.append({
            'PLAYER_ID': player_id,
            'POSITION': info['POSITION'].iloc[0]
        })
        time.sleep(0.5)
    except Exception as e:
        print(f"Error for player {player_id}: {e}")

positions_df = pd.DataFrame(positions)
print(positions_df['POSITION'].value_counts())

POSITION
Guard             43
Forward           23
Guard-Forward     10
Center             8
Forward-Guard      4
Forward-Center     4
Center-Forward     2
Name: count, dtype: int64


In [11]:
# Simplify positions to three categories
position_map = {
    'Guard': 'Guard',
    'Guard-Forward': 'Guard',
    'Forward-Guard': 'Guard',
    'Forward': 'Forward',
    'Forward-Center': 'Forward',
    'Center-Forward': 'Forward',
    'Center': 'Center'
}

positions_df['POSITION_simplified'] = positions_df['POSITION'].map(position_map)

# Merge into all_candidates
all_candidates = all_candidates.merge(
    positions_df[['PLAYER_ID', 'POSITION_simplified']],
    on='PLAYER_ID',
    how='left'
)

print("Missing positions:", all_candidates['POSITION_simplified'].isna().sum())
print(all_candidates['POSITION_simplified'].value_counts())

Missing positions: 0
POSITION_simplified
Guard      62
Forward    30
Center      8
Name: count, dtype: int64


In [12]:
# Pull draft history to access draft position 
draft = drafthistory.DraftHistory().get_data_frames()[0]
print(draft.columns.tolist())

all_candidates = all_candidates.merge(
    draft[['PERSON_ID', 'OVERALL_PICK']].rename(columns={'PERSON_ID': 'PLAYER_ID'}),
    on='PLAYER_ID',
    how='left'
)

all_candidates['OVERALL_PICK'] = all_candidates['OVERALL_PICK'].fillna(61)

print("Missing draft picks:", all_candidates['OVERALL_PICK'].isna().sum())
print("Undrafted players (pick 61):", (all_candidates['OVERALL_PICK'] == 61).sum())

['PERSON_ID', 'PLAYER_NAME', 'SEASON', 'ROUND_NUMBER', 'ROUND_PICK', 'OVERALL_PICK', 'DRAFT_TYPE', 'TEAM_ID', 'TEAM_CITY', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'ORGANIZATION', 'ORGANIZATION_TYPE', 'PLAYER_PROFILE_FLAG']
Missing draft picks: 0
Undrafted players (pick 61): 10


In [13]:
# Pull following season team to determine if player stayed on same team or switched
print(all_candidates.columns.tolist())

all_candidates['change_team'] = all_candidates['TEAM_ID_post'] != all_candidates['TEAM_ID_following']
print(all_candidates[all_candidates['change_team'] == True][['season', 'change_team', 'PLAYER_NAME_post', 'TEAM_ABBREVIATION_post', 'TEAM_ABBREVIATION_following']].to_string())

['PLAYER_ID', 'PLAYER_NAME_post', 'TEAM_ID_post', 'TEAM_ABBREVIATION_post', 'AGE_post', 'GP_post', 'MIN_post', 'PIE_post', 'USG_PCT_post', 'PLAYER_NAME_pre', 'TEAM_ID_pre', 'TEAM_ABBREVIATION_pre', 'AGE_pre', 'GP_pre', 'MIN_pre', 'PIE_pre', 'USG_PCT_pre', 'PIE_previous', 'PIE_following', 'GP_following', 'TEAM_ID_following', 'TEAM_ABBREVIATION_following', 'season', 'avg_wins_3yr', 'PIE_change', 'USG_change', 'PIE_diff', 'POSITION_simplified', 'OVERALL_PICK']
     season  change_team          PLAYER_NAME_post TEAM_ABBREVIATION_post TEAM_ABBREVIATION_following
2   2014-15         True            Chase Budinger                    MIN                         PHX
7   2014-15         True                Jeremy Lin                    LAL                         CHA
11  2014-15         True              Ray McCallum                    SAC                         SAS
14  2014-15         True              Shane Larkin                    NYK                         BKN
16  2014-15         True    

In [14]:
model_cols = [
    'PLAYER_NAME_post', 'season',
    'PIE_diff',            # primary outcome
    'PIE_following',       # secondary outcome
    'PIE_change',          # main predictor
    'PIE_pre',             # baseline control
    'USG_change',          # role expansion
    'AGE_post',            # development trajectory
    'MIN_post',            # playing time control
    'avg_wins_3yr',        # role security
    'POSITION_simplified', # categorical
    'OVERALL_PICK',        # draft status
    'change_team',         # binary team status
]

print(all_candidates[model_cols].info())
print("\nMissing values:")
print(all_candidates[model_cols].isna().sum())
print("\nFirst 5 rows:")
print(all_candidates[model_cols].head().to_string())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PLAYER_NAME_post     100 non-null    str    
 1   season               100 non-null    str    
 2   PIE_diff             100 non-null    float64
 3   PIE_following        100 non-null    float64
 4   PIE_change           100 non-null    float64
 5   PIE_pre              100 non-null    float64
 6   USG_change           100 non-null    float64
 7   AGE_post             100 non-null    float64
 8   MIN_post             100 non-null    float64
 9   avg_wins_3yr         100 non-null    float64
 10  POSITION_simplified  100 non-null    str    
 11  OVERALL_PICK         100 non-null    float64
 12  change_team          100 non-null    bool   
dtypes: bool(1), float64(9), str(3)
memory usage: 9.6 KB
None

Missing values:
PLAYER_NAME_post       0
season                 0
PIE_diff               0
PIE

In [20]:
data_path = Path().resolve().parent / 'data' / 'nba_breakout_candidates.csv'
all_candidates.to_csv(data_path, index=False)